# Structure of Water

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/ch2/2A-water-structure/ch2A-water.ipynb)
:::


In [7]:
#| tags: [remove-cell, remove-stdout]

####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

try:
    import rdkit
except ImportError:
    !pip -q install rdkit
    import rdkit

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    plt.style.use("styles/tufte.mplstyle")
    nproc = 4          # my own computer limit: 4 threads. Yours may vary. Adjust this number to your system's capabilities. Don't include efficiency cores as they are not well-suited for Psi4 calculations.

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output/ch2/2A-structure")
output_dir.mkdir(exist_ok=True)


# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
psi4.set_memory("2 GB")

# 3D pictures: False = static images (as in the book); True = rotatable py3Dmol viewers
INTERACTIVE = True

  Threads set to 4 by Python driver.

  Memory set to   1.863 GiB by Python driver.


**Water** is a small molecule and is a great way to **start** exploring *Psi4*. In this chapter, I will give a very incomplete description of how *Psi4* works and how to use it to enter the **structure** of a molecule and calculate the **energy** of that structure.

```{note} The Invisible Code
:class: dropdown
There is a **block** of code **at the top** if each notebook that is **invisible** in this web book. When **running** the notebook in *Colab* or on your own computer, that cell will be visible and needs to be the **first** block **executed**. This setup block at the top of each notebook **sets the resources** that *Psi4* may use. It will sense the environment and, if you are using *Colab*, the code will **install** required *Python* tools. On your own computer you will have set up your *Python* environment in advance according to the [setup instructions](ch1-setup.md).

In this notebook, the hidden code also decalres the **destination** for all output file (`output/ch2/2A-structure`), sets the **number of processors** to use (2 for *Colab*, 4 for my own computer), and sets **flags** that affect many different cells in the notebook. Here, the flag `INTERACTIVE = False` will make all moelcular images render as **still images**. If you set `INTERACTIVE = True`, then those images will be 3-dimensional **interactive objects** that you can spin around and observe at any angle. The interactive images can be very, **very large** when they include molecular orbitals and other objects. The web book is executed with `INTERACTIVE = False` so that I **don't break** the cloud provider that is serving these pages. You can set it to `True` when running your calculation in *Colab* or on your own machine.
```
Before *Psi4* can calculate anything, we have to tell it **where the atoms are**. In this notebook we describe water in four different ways and check that *Psi4* understands each one as the same molecule.

## What You Will Learn
- How to write a molecule as **Cartesian coordinates** ($xyz$).
- How to write a molecule as **internal coordinates** (a *Z-matrix*) using bond lengths and angles.
- How *Psi4* detects **symmetry** and moves the molecule into a standard orientation.
- How to get a starting structure from a **SMILES string** with *RDKit*.
- How to run a first **single-point energy** and where the output files go.

<!--  
```{note} Settings used in every notebook
:class: dropdown
The setup cell at the top of each notebook sets the resources that *Psi4* may use:

`psi4.set_num_threads(nproc)`
: The number of CPU cores. The house limit is 4 threads. *Colab* gives you 2. Water is so small that extra cores barely help; they matter for larger molecules.

`psi4.set_memory("2 GB")`
: The memory *Psi4* may use for its arrays. The house limit is 4 GB. Water needs far less, but 2 GB is a safe value that also works on *Colab* (about 12 GB total). Alternative: `"500 MB"` is plenty for most of these calculations; larger values only matter for big basis sets or coupled-cluster methods on bigger molecules.

`psi4.set_output_file("output/name.out", False)`
: Where *Psi4* writes its detailed text output. *Psi4* also writes a `.log` file with the same name beside it. `False` means overwrite the file each time; `True` would append. We keep every output in the `output` folder so the notebook folder stays tidy. A file called `timer.dat` may still appear beside the notebook; you can ignore it.
```
-->

## 1. Cartesian Coordinates

*x*, *y*, *z*... That triplet describes the **entire universe**. When we give *Psi4* a set of Cartesian coordinates, we may place the molecule anywhere we like. *Psi4* will then **move** the molecule so that its **center of mass** is at the origin and **rotate** it into a standard orientation that suits its symmetry.

The most direct description lists each atom with its *x*, *y* and *z* position. We can describe the geometry as a short block of plain text. The first line in the text block is the **charge** and **spin multiplicity** (here `0 1`: neutral, all electrons paired, a *singlet*). The next lines are the *xyz* coordinates of the atoms in ångströms.

The **coordinates** in the [code below](#water-coordinates) were written by hand from the experimental structure ($r_{\mathrm{OH}} = 0.958$ Å, $\angle\mathrm{HOH} = 104.5°$). Simple **trigonometry** can calculate the *xy* positions of the hydrogen atoms. We will set the oxygen atom at the origin and keep all atoms in the *xy* plane.

In [2]:
#| label: water-coordinates

# a string of ASCII characters that defines the geometry of a water molecule
water_xyz = """
0 1                                         # charge and multiplicity
O    0.000000    0.000000    0.000000       # atom 1: oxygen
H    0.586700    0.757900    0.000000       # atom 2: hydrogen
H    0.586700   -0.757900    0.000000       # atom 3: hydrogen
"""

mol = psi4.geometry(water_xyz)     # make a Psi4 molecule object from the string
mol.update_geometry()              # make Psi4 process the input now.

print("Point group:", mol.schoenflies_symbol()) # print the point group
print(mol.save_string_xyz_file())  # print the geometry in XYZ format

Point group: c2v
3

 O    0.000000000000    0.000000000000   -0.065660456158
 H    0.000000000000   -0.757900000000    0.521039543842
 H   -0.000000000000    0.757900000000    0.521039543842



Observe: the **coordinates** printed by *Psi4* in the output from the code above are **not** the ones we typed. *Psi4* has

1. determined that water belongs to the *C<sub>2v</sub>* **point group**. *Psi4* uses that symmetry to label orbitals and vibrations and to speed up some calculations.
2. moved the **centre of mass** to the origin.
3. **rotated** the molecule so its symmetry elements line up with the axes. You can see that *Psi4* sets the atoms in the *z* axis as the principle *C<sub>2</sub>* axis when using the *C<sub>2v</sub>* point group. 

```{figure} images/ch2/A-Water-xyz2.png
:name: fig-water
:alt: Ball-and-stick model of water
:width: 500px
:align: center

The coordinates as entered (left) were rotated and moved by *Psi4* to place the $C_2$ axis along the $z$-axis and place the center the mass of the molecule at the origin (right).
```

### Symmetry

Symmetry is beautiful. It also helps with math. If a molecule has **symmetry**, the calculation can be **less expensive**. A molecule that is known to possess 2-fold symmetry may be half as difficult to analyze because we may only need half the math.

**Water** has a *C<sub>2</sub>* axis. There is no perpendicular *C<sub>2</sub>* axis so we remain in the *C* group rather than rising to *D*. There is no *&sigma;<sub>h</sub>* but there are two *&sigma;<sub>v</sub>* planes. Therefore water is in the *C<sub>2v</sub>* [point group](https://en.wikipedia.org/wiki/Point_group). *Psi4* placed the **primary axis** along the *z*-axis and the atoms in the *yz* plane. See {numref}`fig-water`.

*Psi4* only does its math within the point group *D<sub>2h</sub>* and its **subgroups** (*C<sub>1</sub>*, *C<sub>i</sub>*, *C<sub>s</sub>*, *C<sub>2</sub>*, *C<sub>2v</sub>*, *C<sub>2h</sub>*, *D<sub>2</sub>* and *D<sub>2h</sub>*). These are the [Abelian](https://en.wikipedia.org/wiki/Abelian_group) point groups {cite:p}`10.1021/acs.jpca.6c00998`, where the symmetry operations behave nicely with each other. Water is *C<sub>2v</sub>*, so *Psi4* can use all of its symmetry. Ammonia is *C<sub>3v</sub>*, and *Psi4* will do its math in *C<sub>s</sub>*, the largest subgroup of *C<sub>3v</sub>* that it can use. *Psi4* will report both the **full** point group and the **computational** point group.

### Querying the Structure

If we performed calculations to optimize the structure, we might have new bond lengths and angles on the molecule object. But it is difficult to get that structural information from a block of *xyz* data. This notebook loaded a set of *Python* code that defined several helpful functions when the first code block (invisible in this web book) was run. Those helper functions are in a file named `psi4_helpers.py`

The [code below](#query-bond-lengths-angles) will calculate bond lengths and angles from *xyz* data.  

In [3]:
#| label: query-bond-lengths-angles

name = "water_xyz"

# Calculate bond lengths and angles using the Psi4 helper functions
r = ph.bond_length(mol, 1, 2)    # bond length between atoms 1 and 2
a = ph.bond_angle(mol, 2, 1, 3)  # bond angle between atoms 2, 1, and 3

print(f"{name:10s}  r(O-H) = {r:.4f} Å   angle(H-O-H) = {a:.2f}°")

water_xyz   r(O-H) = 0.9585 Å   angle(H-O-H) = 104.51°


## Internal Coordinates

Trigonometry is hard! I had to look up *sine* and *cosine*. I won't be able to handle it when we need to describe atoms positions in **all three dimensions**. For many of our exercises, we will be building molecules using **internal coordinates**. These coordinates define atomic positions **relative** to atoms that have already been placed. No *x*, *y* or *z* will be required. A **Z-matrix** is a defined format for listing these internal coordinates.



### The Z-matrix with Water

Here is a Z-matrix for the molecule water. 


```{code-block} text
:label: water-inp
:caption: water.inp
:linenos: 

H
O  1  0.958
H  2  0.958  1  104.5
```
| Line | Notes |
| :---- | :---- |
|  1 |	atom 1 is hydrogen |
|  2 |	atom 2 is oxygen. The 2,1 distance is 0.958 &#8491;. |
|  3 |	atom 3 is hydrogen. The 3,2 distance is 0.958 &#8491;. The 3,2,1 bond angle is 104.5&deg; |


It is interpreted in {numref}`fig-zmatrix`.
```{figure} images/ch2/A-Zmatrix.png
:name: fig-zmatrix
:alt: Z0matrix for water
:width: 500px
:align: center

The Z-matrix for water presented graphically.
```
### Using Variables in a Z-matrix
For a molecule to have *C<sub>2v</sub>* symmetry, the two O&ndash;H bond lengths **must** be the same. One way that we can make sure that this is true is to use **variables** in the Z-matrix. Below, the bond length is given the name `r` and the bond angle the name `a`. The values of the variables are given at the **end** of the molecule description. Both bonds use the same variable, so they are guaranteed to be identical. This is a useful way to define **symmetry** in an input structure.

The [code below](#water-zmatrix) sends a **text string** that represents a Z-matrix into *Pi4* to create a **molecule object**. We can then get the object to **report** the molecule in Cartesian coordinates and as a Z-matrix. 

Inputting structures in the Z-matrix format is useful. We can easily **identify** bond **lengths** and **angles** in the Z-matrix data. Soon we will be performing calculations to optimize **structure**. After the optimization, the molecule object will contain the new Z-matrix for the optimized structure. We can **read** the bond lengths and angles right from the text **printout** of the Z-matrix. The structure must have been **input** via a Z-matrix for *Psi4* to keep a Z-matrix representation in the molecule object.

In [4]:
#| label: water-zmatrix

# a string of ASCII characters that defines a Z-matrix for a water molecule.
water_zmat = """
0 1           # charge and multiplicity
H             # atom 1: hydrogen
O 1 r         # atom 2: oxygen, bonded to atom 1 with bond length r
H 2 r 1 a     # atom 3: hydrogen, bonded to atom 2 with bond length r and angle a with respect to atom 1

r = 0.958     # bond length in Angstroms
a = 104.5     # bond angle in degrees
"""

mol = psi4.geometry(water_zmat)   # make a Psi4 molecule object from the string
mol.update_geometry()             # make Psi4 process the input now

print("Point group:", mol.schoenflies_symbol())  # the point group 

print("\nCartesian coordinates:") # the '\n' is a newline character to separate the output
print(mol.save_string_xyz_file()) # print the geometry in XYZ format

print("\nZ-matrix:")
print(mol.create_psi4_string_from_molecule())  # print the Z-matrix 

Point group: c2v

Cartesian coordinates:
3

 H    0.000000000000    0.757480611647   -0.520865616165
 O    0.000000000000    0.000000000000    0.065638538108
 H    0.000000000000   -0.757480611647   -0.520865616165


Z-matrix:
    units Angstrom
    0 1
    --
    0 1
    H           
    O                 1           r
    H                 2           r      1           a

    a         =  104.5000000000
    r         =    0.9580000000




You can see that, in the reported Cartesian coordinates [above](#water-zmatrix), *Psi4* has positioned the water molecule with the *z*-axis being the principle C<sub>2</sub> axis and all the atoms are in the *yz* plane. However, look closely and you will see that the water molecule is upside-down relative to the structure that was generated using the *xyz* [Cartesian coordinates](#water-coordinates) as input. This makes no difference, but it is curious. 

## Visualize the Molecule

We can visualize the molecule using the *py3Dmol* package in *Python*. The function `ph.show_molecule()` will display the structure in ball and stick style. You can choose fixed pictures or 3D models that you can rotate and zoom with the mouse. You can choose by setting `INTERACTIVE = True` or `INTERACTIVE = False` in the first cell and rerun the notebook to change the output.


In [8]:
ph.show_molecule(mol, interactive=INTERACTIVE)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4. A Starting Structure from *RDKit*

For bigger molecules, typing coordinates is slow and error-prone. *RDKit* can build a reasonable 3D structure from a **SMILES** string using a force field (MMFF) optimization. This is only a **starting guess**; we will improve it with *Psi4* in the next notebook.

*RDKit* is a powerful **cheminformatics** package and provide tools for describing, building and analyzing molecular structures. The [code below](#water-rdkit) will take a SMILES string and get an structure for water that has been optimized using a **molecular mechanics** computation. You can see in the output that it is slightly **different** than the values that we used previously.

In [18]:
#| label: water-rdkit

from rdkit import Chem
from rdkit.Chem import AllChem

rdmol = Chem.MolFromSmiles("O")      # SMILES for water is just "O". Hydrogens are implied in SMILES, so we add them explicitly with AddHs()
rdmol = Chem.AddHs(rdmol)            # SMILES for water is just "O". Hydrogens are implied in SMILES, so we add them explicitly with AddHs()
AllChem.EmbedMolecule(rdmol, randomSeed=42)      # randomSeed makes the result repeatable
AllChem.MMFFOptimizeMolecule(rdmol)              # quick force-field clean-up

xyz_lines = Chem.MolToXYZBlock(rdmol).splitlines()[2:] # drop the atom count and comment lines
charge = Chem.GetFormalCharge(rdmol)                   # get the formal charge of the molecule

water_rdkit = f"{charge} 1\n" + "\n".join(xyz_lines) # make a text string for the xyz coordinates with charge and multiplicity
mol = psi4.geometry(water_rdkit)             # make a Psi4 molecule object from the string
mol.update_geometry()                        # make Psi4 process the input now

r = ph.bond_length(mol, 1, 2)    # bond length between atoms 1 and 2
a = ph.bond_angle(mol, 2, 1, 3)  # bond angle between atoms 2, 1, and 3

print("Cartessian coordinates from RDKit:\n")
print(water_rdkit) # print XYZ text that was sent into psi4.geometry()
print("\nCalculated bond lengths and angles from RDKit geometry:\n")
print(f"r(O-H) = {r:.4f} Å   angle = {a:.2f}°")

Cartessian coordinates from RDKit:

0 1
O      0.007827    0.397738    0.000000
H     -0.767234   -0.183847    0.000000
H      0.759407   -0.213891    0.000000

Calculated bond lengths and angles from RDKit geometry:

r(O-H) = 0.9690 Å   angle = 103.98°


```{note} Other ways to get a starting structure
:class: dropdown
- **PubChem:** `psi4.geometry("pubchem:water")` downloads a structure by name. It needs an internet connection and sometimes fails when PubChem is busy, so it is less reliable in a class setting.
- **Draw it:** programs such as *Avogadro* or *IQmol* let you draw a molecule and export an `.xyz` file.
- **From a file:** read an `.xyz` file with *Python* and pass the text to `psi4.geometry()` (remember to add the charge/multiplicity line).
```